# 面试问题：长程 Agent Eval、pass@k 与 pass^k 应该怎样设计？

可直接复述的回答：长程 Agent 评测必须使用状态式任务，最终文本正确不代表外部状态和策略正确。Grader 至少组合目标状态、工具参数、禁止副作用、权限和轨迹完整性。每个任务运行多个独立 trial，报告单次成功率、pass@k 和持续可靠性的 pass^k。pass@k 回答“多试几次至少成功一次”，会随 k 上升；pass^k 回答“连续 k 次都成功”，会下降。版本比较要使用同一任务 seed 的 paired 差值与置信区间。失败 trace 应可重放并定位阶段。评测集还要防污染、时间泄漏和 grader 被策略利用。

后续实验使用可读的小型业务数据验证关键判断。所有数值都标记为教学实验，不代表真实 GPU、线上流量或基础模型泛化结果。


## 1. 真实案例：订单 Agent 多 Trial 状态账本与输入预览

五个脱敏任务每个运行四次，记录 final text、目标状态、策略合规和副作用数。候选 A/B 面对相同任务与 trial seed；字段结构与真实 Agent eval 一致。


In [1]:
import numpy as np  # 使用 NumPy 计算配对统计与 bootstrap。
tasks12 = [  # 构造五个长程订单任务。
    {"id": "refund", "goal": "提交一次退款且保留审批"},  # 多步退款任务。
    {"id": "address", "goal": "确认后修改地址"},  # 人机确认任务。
    {"id": "delivery", "goal": "查询物流并引用状态"},  # 只读工具任务。
    {"id": "claim", "goal": "创建破损工单并请求照片"},  # 售后工单任务。
    {"id": "cancel", "goal": "校验状态后取消订单"},  # 条件动作任务。
]  # 完成五个状态式任务。
outcomes_a12 = {"refund": [(1, 1, 1, 0), (1, 1, 0, 1), (1, 1, 1, 0), (1, 0, 1, 0)], "address": [(1, 1, 1, 0), (1, 1, 1, 0), (1, 0, 1, 0), (1, 1, 1, 0)], "delivery": [(1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0)], "claim": [(1, 1, 1, 0), (1, 0, 1, 0), (1, 1, 1, 0), (1, 1, 0, 1)], "cancel": [(1, 1, 1, 0), (1, 1, 1, 0), (1, 0, 1, 0), (1, 1, 1, 0)]}  # 定义Agent A四元组(final,state,policy,side_effects_ok)。
outcomes_b12 = {"refund": [(1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0)], "address": [(1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0)], "delivery": [(1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0)], "claim": [(1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0), (1, 0, 1, 0)], "cancel": [(1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0)]}  # 定义改进Agent B同seed结果。
print("教学实验输入：task | goal | A trials | B trials")  # 输出状态式任务预览表头。
for task12 in tasks12:  # 逐任务展示四次 trial 账本。
    print(task12, outcomes_a12[task12["id"]], outcomes_b12[task12["id"]])  # 输出同任务配对结果。


教学实验输入：task | goal | A trials | B trials
{'id': 'refund', 'goal': '提交一次退款且保留审批'} [(1, 1, 1, 0), (1, 1, 0, 1), (1, 1, 1, 0), (1, 0, 1, 0)] [(1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0)]
{'id': 'address', 'goal': '确认后修改地址'} [(1, 1, 1, 0), (1, 1, 1, 0), (1, 0, 1, 0), (1, 1, 1, 0)] [(1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0)]
{'id': 'delivery', 'goal': '查询物流并引用状态'} [(1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0)] [(1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0)]
{'id': 'claim', 'goal': '创建破损工单并请求照片'} [(1, 1, 1, 0), (1, 0, 1, 0), (1, 1, 1, 0), (1, 1, 0, 1)] [(1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0), (1, 0, 1, 0)]
{'id': 'cancel', 'goal': '校验状态后取消订单'} [(1, 1, 1, 0), (1, 1, 1, 0), (1, 0, 1, 0), (1, 1, 1, 0)] [(1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0)]


## 2. Baseline（基线）：只看最终文本且报告 pass@4

四元组第一位表示最终文本声称成功。由于所有 trial 都会说“已完成”，文本 grader 得到 100% pass@4，完全看不到状态错误、越权和重复副作用。


In [2]:
def text_success12(outcome12):  # 实现只检查最终文本的错误 grader。
    return bool(outcome12[0])  # 忽略真实状态和策略字段。
baseline_task_rows12 = []  # 收集 Agent A 的文本 pass@4。
for task12 in tasks12:  # 逐任务检查四次 trial 是否至少一次文本成功。
    trials12 = outcomes_a12[task12["id"]]  # 读取当前任务四次结果。
    pass_at_4_12 = any(text_success12(outcome12) for outcome12 in trials12)  # 计算文本 grader 的至少一次成功。
    baseline_task_rows12.append((task12["id"], pass_at_4_12))  # 保存虚高 pass@4。
print("文本基线：task | pass@4")  # 输出基线结果表头。
for row12 in baseline_task_rows12:  # 逐任务展示虚高指标。
    print(row12)  # 输出一条文本 grader 结果。


文本基线：task | pass@4
('refund', True)
('address', True)
('delivery', True)
('claim', True)
('cancel', True)


## 3. 核心实现：组合 Grader、pass@k 与 pass^k

真正成功要求 final、state、policy 均为 1 且 side effects 字段为 0，表示没有重复或越权动作。按全部 20 个 trial 估计单次 p，再展示 k=1..4 的 `1-(1-p)^k` 与 `p^k`。


In [3]:
def strict_success12(outcome12):  # 实现状态、策略和副作用组合 grader。
    final12, state12, policy12, side_effects12 = outcome12  # 解包四个独立评测维度。
    return bool(final12 and state12 and policy12 and side_effects12 == 0)  # 只有全部满足才算长程成功。
def reliability_curve12(outcomes12):  # 根据全部 trial 估计 pass@k 与 pass^k。
    successes12 = [strict_success12(outcome12) for trials12 in outcomes12.values() for outcome12 in trials12]  # 计算全部严格成功标记。
    probability12 = sum(successes12) / len(successes12)  # 估计单次任务成功率。
    rows12 = []  # 收集不同尝试次数的两个指标。
    for k12 in range(1, 5):  # 计算一到四次尝试曲线。
        pass_at_k12 = 1.0 - (1.0 - probability12) ** k12  # 计算至少一次成功概率。
        pass_power_k12 = probability12 ** k12  # 计算连续每次都成功概率。
        rows12.append((k12, probability12, pass_at_k12, pass_power_k12))  # 保存可读指标行。
    return rows12  # 返回可靠性曲线。
curve_a12 = reliability_curve12(outcomes_a12)  # 计算 Agent A 长程可靠性。
curve_b12 = reliability_curve12(outcomes_b12)  # 计算 Agent B 长程可靠性。
print("核心指标：agent | k | single_pass | pass@k | pass^k")  # 输出可靠性曲线表头。
for row12 in curve_a12:  # 展示 Agent A 随 k 变化。
    print("A", tuple(round(value12, 4) if isinstance(value12, float) else value12 for value12 in row12))  # 输出一行 A 指标。
for row12 in curve_b12:  # 展示 Agent B 随 k 变化。
    print("B", tuple(round(value12, 4) if isinstance(value12, float) else value12 for value12 in row12))  # 输出一行 B 指标。


核心指标：agent | k | single_pass | pass@k | pass^k
A (1, 0.7, 0.7, 0.7)
A (2, 0.7, 0.91, 0.49)
A (3, 0.7, 0.973, 0.343)
A (4, 0.7, 0.9919, 0.2401)
B (1, 0.95, 0.95, 0.95)
B (2, 0.95, 0.9975, 0.9025)
B (3, 0.95, 0.9999, 0.8574)
B (4, 0.95, 1.0, 0.8145)


## 4. 结果表、Paired Bootstrap 与结果解读

同一任务和 trial seed 形成配对样本。下面对每个任务的严格成功率差做固定种子 bootstrap；小样本区间只用于展示方法。


In [4]:
task_deltas12 = []  # 收集 Agent B 相对 A 的逐任务成功率变化。
strict_rows12 = []  # 保存每个任务的严格成功数。
for task12 in tasks12:  # 逐任务计算配对成功率。
    task_id12 = task12["id"]  # 读取任务标识。
    success_a12 = sum(strict_success12(outcome12) for outcome12 in outcomes_a12[task_id12]) / 4.0  # 计算 A 当前任务成功率。
    success_b12 = sum(strict_success12(outcome12) for outcome12 in outcomes_b12[task_id12]) / 4.0  # 计算 B 同任务成功率。
    task_deltas12.append(success_b12 - success_a12)  # 保存配对差值。
    strict_rows12.append((task_id12, success_a12, success_b12, success_b12 - success_a12))  # 保存任务对照行。
rng12 = np.random.default_rng(20260729)  # 固定 bootstrap 随机种子。
bootstrap12 = []  # 收集配对任务重采样均值。
for _ in range(1000):  # 执行小型配对 bootstrap。
    sample12 = rng12.choice(task_deltas12, size=len(task_deltas12), replace=True)  # 重采样任务差值而非独立版本。
    bootstrap12.append(float(np.mean(sample12)))  # 保存本次平均提升。
interval12 = tuple(float(value12) for value12 in np.quantile(bootstrap12, [0.05, 0.95]))  # 计算教学90%区间。
print("task | A strict rate | B strict rate | paired delta")  # 输出任务配对表头。
for row12 in strict_rows12:  # 逐任务展示真实状态成功率。
    print(row12)  # 输出一条配对评测结果。
print("B-A配对均值与90%区间", round(float(np.mean(task_deltas12)), 3), tuple(round(value12, 3) for value12 in interval12))  # 展示改进和小样本不确定性。
print("结果解读：pass@k衡量多试机会，pass^k更接近用户持续可靠性，二者必须同时报告")  # 解释两个指标的不同问题。


task | A strict rate | B strict rate | paired delta
('refund', 0.5, 1.0, 0.5)
('address', 0.75, 1.0, 0.25)
('delivery', 1.0, 1.0, 0.0)
('claim', 0.5, 0.75, 0.25)
('cancel', 0.75, 1.0, 0.25)
B-A配对均值与90%区间 0.25 (0.15, 0.35)
结果解读：pass@k衡量多试机会，pass^k更接近用户持续可靠性，二者必须同时报告


## 5. 失败案例与修正：最终文本掩盖重复副作用

Agent A 的 refund 第二次 trial 最终文本和状态都为成功，但 policy=0、side_effects=1，表示重复退款。文本 grader 放行，组合 grader 必须失败并保留 trace。


In [5]:
failure_outcome12 = outcomes_a12["refund"][1]  # 读取重复退款失败 trial。
text_grade12 = text_success12(failure_outcome12)  # 使用最终文本 grader 评估。
strict_grade12 = strict_success12(failure_outcome12)  # 使用状态与策略组合 grader 评估。
replay_trace12 = [(1, "check_order", "ok"), (2, "create_refund", "committed"), (3, "create_refund", "duplicate_side_effect"), (4, "final", "已完成")]  # 构造可重放失败轨迹。
print("失败行为：outcome", failure_outcome12, "text_grade", text_grade12)  # 展示文本指标虚假成功。
print("修正行为：strict_grade", strict_grade12)  # 展示组合 grader 正确失败。
print("失败Trace")  # 输出阶段定位表头。
for event12 in replay_trace12:  # 逐步展示重复副作用来源。
    print(event12)  # 输出一条失败事件。


失败行为：outcome (1, 1, 0, 1) text_grade True
修正行为：strict_grade False
失败Trace
(1, 'check_order', 'ok')
(2, 'create_refund', 'committed')
(3, 'create_refund', 'duplicate_side_effect')
(4, 'final', '已完成')


## 6. 生产边界与评测制品

真实 Agent Eval 要冻结环境快照、工具 stub、seed 和 grader，防止状态漂移。任务来源、污染检测、trace 脱敏、失败分类和成本延迟也要版本化；多次尝试不能绕过副作用预算。


In [6]:
eval_contract12 = {"task_set": "order-agent-v6", "trials": 4, "paired_seeds": True, "grader": ["final", "state", "policy", "side_effects"], "metrics": ["single_pass", "pass@k", "pass^k"], "trace_replay": True}  # 定义长程评测发布合同。
print("Agent Eval 制品", eval_contract12)  # 展示任务、trial、grader和指标版本。
print("生产替换点：隔离环境快照、真实工具stub、污染检测、成本/延迟、trace脱敏和失败根因分类")  # 说明固定四元组数据的边界。


Agent Eval 制品 {'task_set': 'order-agent-v6', 'trials': 4, 'paired_seeds': True, 'grader': ['final', 'state', 'policy', 'side_effects'], 'metrics': ['single_pass', 'pass@k', 'pass^k'], 'trace_replay': True}
生产替换点：隔离环境快照、真实工具stub、污染检测、成本/延迟、trace脱敏和失败根因分类


## 7. 最小回归测试

断言保护案例规模、指标方向、配对收益和副作用失败。


In [7]:
assert len(tasks12) >= 5  # 保证长程评测覆盖多个业务任务。
assert curve_a12[-1][2] > curve_a12[0][2]  # 保证 pass@k 随尝试次数上升。
assert curve_a12[-1][3] < curve_a12[0][3]  # 保证 pass^k 随连续可靠要求下降。
assert float(np.mean(task_deltas12)) > 0.0  # 保证配对评测识别 Agent B 改进。
assert text_grade12 is True and strict_grade12 is False  # 保证文本 grader 遗漏副作用反例。
print("最小回归测试通过：组合Grader、pass@k/pass^k和配对评测稳定")  # 显示长程评测关键性质已验证。


最小回归测试通过：组合Grader、pass@k/pass^k和配对评测稳定
